# Fase 3 — Programación Orientada a Objetos

**Asignatura:** MCDI500 — Introducción a la Programación  
**Proyecto:** Detección de Anomalías y Posibles Fraudes en Permisos de Circulación Vehicular  
**Stack:** Python 3.14 · pandas · scikit-learn · loguru

# Proyecto de Magíster: Detección de Anomalías y Posibles Fraudes en Permisos de Circulación

## Problemática
En la administración municipal, el cobro de los Permisos de Circulación Vehicular depende fuertemente de datos ingresados manualmente y tasaciones. Debido a la falta de validación estandarizada en los sistemas locales, existen múltiples errores de digitación, inconsistencias de formato y, potencialmente, **fraudes y evasiones** (por ejemplo, vehículos de alto valor comercial registrados con características alteradas para pagar menos, o modificaciones vehiculares no declaradas). Esta desestructuración de la información genera ineficiencias en la gestión pública y grandes pérdidas de ingresos municipales.

## Objetivo General
Diseñar e implementar soluciones algorítmicas en Python aplicadas al proyecto transversal, integrando principios de programación estructurada, recursiva y orientada a objetos para construir código modular, reutilizable y escalable.

## Configuración del Entorno

Importación de librerías externas e internas. `loguru` reemplaza el módulo `logging` estándar entregando timestamps, niveles de color y formato estructurado sin configuración adicional.

In [1]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from loguru import logger
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

## Arquitectura de Clases


Cada clase tiene **una sola responsabilidad** (alta cohesión) y expone un único método público orquestador que ejecuta los pasos privados en secuencia (bajo acoplamiento entre módulos).

### Clase `Explorador`

Analiza el dataset en crudo: dimensiones, tipos de datos, nulos, duplicados, tipos mixtos, anomalías de texto y coherencia de fechas. Todos los análisis son métodos privados (`_`); `explorar()` los ejecuta en orden y es el único punto de entrada público.

In [2]:
class Explorador:
    """Clase para realizar un análisis exploratorio inicial del dataset,
    enfocándose en la estructura, calidad y posibles anomalías.
    Cada método privado se encarga de un aspecto específico del análisis,
    y el método público 'explorar' los ejecuta en secuencia."""

    def __init__(self, df):
        self.df = df

    def _find_col(self, *names):
        """Retorna el primer nombre de columna que exista en el df,
        usando comparación case-insensitive. Funciona antes y después de codificar()."""
        lower_map = {c.lower(): c for c in self.df.columns}
        for name in names:
            found = lower_map.get(name.lower())
            if found:
                return found
        return None

    def _explorar_estructura(self):
        logger.info("Estructura del dataset")
        print(f"Dimensiones: {self.df.shape[0]} filas x {self.df.shape[1]} columnas\n")
        display(self.df.head(10))
        print("\n--- Tipos de datos ---")
        display(self.df.dtypes.rename('tipo').to_frame())
        print("\n--- Estadísticas descriptivas ---")
        display(self.df.describe())

    def _analizar_categorias(self, columnas=None):
        logger.info("Frecuencias por columna categórica")
        if columnas is None:
            columnas = self.df.select_dtypes(include='object').columns.tolist()
        for col in columnas:
            print(f"\n{col}:")
            print(self.df[col].value_counts(dropna=False))
            print("-" * 30)

    def _generar_reporte_nulos(self):
        logger.info("Reporte de valores nulos")
        reporte = []
        for col in self.df.columns:
            nulos = self.df[col].isnull().sum()
            reporte.append({
                'Columna': col,
                'Tipo': str(self.df[col].dtype),
                'Nulos': nulos,
                '% Nulos': f"{nulos / len(self.df) * 100:.2f}%"
            })
        df_reporte = pd.DataFrame(reporte)
        con_nulos = df_reporte[df_reporte['Nulos'] > 0].sort_values('Nulos', ascending=False)
        display(con_nulos)
        return df_reporte

    def _detectar_duplicados(self):
        n = self.df.duplicated().sum()
        if n > 0:
            logger.warning(f"Filas totalmente duplicadas: {n}")
            col_id = self._find_col('_id', 'id')
            df_dup = self.df[self.df.duplicated(keep=False)]
            display(df_dup.sort_values(col_id).head(10) if col_id else df_dup.head(10))
        else:
            logger.success("Sin filas duplicadas")
        return n

    def _detectar_tipos_mixtos(self, columnas_numericas):
        logger.info("Detección de tipos mixtos en columnas numéricas")
        for col in [c for c in columnas_numericas if c in self.df.columns]:
            temp = pd.to_numeric(self.df[col], errors='coerce')
            mascara = self.df[col].notnull() & temp.isnull()
            if mascara.any():
                logger.warning(f"'{col}': {self.df[mascara][col].unique()} ({mascara.sum()} filas con valores no numéricos)")
            else:
                logger.success(f"'{col}': OK")

    def _analizar_anomalias_texto(self):
        logger.info("Longitud máxima en columnas de texto")
        cols_texto = [self._find_col(c) for c in ['TipoVehiculo', 'Marca', 'Modelo']]
        cols_texto = [c for c in cols_texto if c]
        for col in cols_texto:
            max_len = self.df[col].astype(str).map(len).max()
            if max_len > 50:
                logger.warning(f"'{col}': {max_len} caracteres (supera umbral)")
                display(self.df[self.df[col].astype(str).map(len) > 50][[col]].head())
            else:
                logger.success(f"'{col}': {max_len} caracteres")
        col_tipo = self._find_col('TipoVehiculo')
        col_cil  = self._find_col('Cilindrada')
        if col_tipo and col_cil:
            logger.info(f"Cilindrada promedio por {col_tipo}")
            resumen = self.df.groupby(col_tipo)[col_cil].agg(['mean', 'max', 'min'])
            display(resumen.sort_values('mean', ascending=False))

    def _validar_coherencia_fechas(self):
        col_fe    = self._find_col('Fecha_Emision')
        col_fv    = self._find_col('Fecha_Vencimiento')
        col_id    = self._find_col('_id', 'id')
        col_anio  = self._find_col('Ano_Fabricacion')
        col_marca = self._find_col('Marca')
        col_mod   = self._find_col('Modelo')

        if not col_fe or not col_fv:
            logger.warning("Columnas de fecha no encontradas — validación omitida")
            return 0

        fe = pd.to_datetime(self.df[col_fe], errors='coerce')
        fv = pd.to_datetime(self.df[col_fv], errors='coerce')
        mascara = fe > fv
        n = mascara.sum()
        if n > 0:
            logger.warning(f"Registros con {col_fe} > {col_fv}: {n}")
            cols_disp = [c for c in [col_id, col_fe, col_fv, col_anio, col_marca, col_mod] if c]
            display(self.df[mascara][cols_disp].head(10))
            if col_anio:
                plt.figure(figsize=(10, 5))
                sns.histplot(self.df[mascara][col_anio], bins=15, kde=True, color='orange')
                plt.title('Año de Fabricación — Registros con Fechas Inconsistentes')
                plt.xlabel('Año de Fabricación')
                plt.ylabel('Frecuencia')
                plt.grid(axis='y', linestyle='--', alpha=0.7)
                plt.tight_layout()
                plt.show()
        else:
            logger.success("Coherencia de fechas: OK")
        return n

    def explorar(self, columnas_numericas=None):
        if columnas_numericas is None:
            columnas_numericas = self.df.select_dtypes(include='number').columns.tolist()
        self._explorar_estructura()
        self._analizar_categorias()
        self._generar_reporte_nulos()
        self._detectar_duplicados()
        self._detectar_tipos_mixtos(columnas_numericas)
        self._analizar_anomalias_texto()
        self._validar_coherencia_fechas()

### Jerarquía de Transformadores: Herencia y Polimorfismo


`Transformador` es una clase abstracta (`ABC`) que define el contrato: toda subclase debe implementar `aplicar(df) → DataFrame`. Esto garantiza una interfaz uniforme sin importar la estrategia concreta.

```
Transformador  (ABC — contrato)
├── ImputarModa          → rellena con la moda de la columna
├── ImputarMediana       → rellena con mediana, opcionalmente agrupando por categoría
├── ImputarValorFijo     → rellena con un valor literal definido por el usuario
├── EscalarZScore        → StandardScaler  (media=0, std=1)
└── EscalarMinMax        → MinMaxScaler    (rango [0, 1])
```

El `Limpiador` y el `Preprocesador` iteran `for t in transformadores: df = t.aplicar(df)` sin conocer el tipo concreto — ese es el **polimorfismo**.

In [3]:
from abc import ABC, abstractmethod
from sklearn.preprocessing import StandardScaler, MinMaxScaler


class Transformador(ABC):
    """Contrato común para todos los transformadores del pipeline.

    Cualquier clase que herede debe implementar aplicar(self, df)
    y retornar el DataFrame modificado.
    """

    @abstractmethod
    def aplicar(self, df):
        """Aplica la transformación sobre df y lo retorna."""
        ...


# ── Imputación ───────────────────────────────────────────────────────────────

class ImputarModa(Transformador):
    """Imputa nulos de columnas categóricas con la moda."""

    def __init__(self, columnas):
        self.columnas = columnas if isinstance(columnas, list) else [columnas]

    def aplicar(self, df):
        for col in self.columnas:
            if col in df.columns:
                df[col] = df[col].fillna(df[col].mode()[0])
                logger.info(f"ImputarModa: '{col}' imputada con moda")
        return df


class ImputarMediana(Transformador):
    """Imputa nulos numéricos con la mediana, opcionalmente agrupando por otra columna."""

    def __init__(self, columna, grupo=None):
        self.columna = columna
        self.grupo   = grupo

    def aplicar(self, df):
        if self.grupo and self.grupo in df.columns:
            df[self.columna] = (
                df.groupby(self.grupo)[self.columna]
                .transform(lambda x: x.fillna(x.median()))
            )
        df[self.columna] = df[self.columna].fillna(df[self.columna].median())
        logger.info(f"ImputarMediana: '{self.columna}' imputada con mediana")
        return df


class ImputarValorFijo(Transformador):
    """Imputa nulos con un valor fijo por columna."""

    def __init__(self, columnas_valores: dict):
        self.columnas_valores = columnas_valores

    def aplicar(self, df):
        for col, valor in self.columnas_valores.items():
            if col in df.columns:
                df[col] = df[col].fillna(valor)
                logger.info(f"ImputarValorFijo: '{col}' → '{valor}'")
        return df



class ConsolidarCategorias(Transformador):
    """Reemplaza alias en una columna categórica mediante un mapeo."""

    def __init__(self, columna: str, mapeo: dict):
        self.columna = columna
        self.mapeo   = mapeo

    def aplicar(self, df):
        if self.columna in df.columns:
            df[self.columna] = df[self.columna].replace(self.mapeo)
            logger.info(f"ConsolidarCategorias: '{self.columna}' consolidada")
        return df

# ── Escalado ─────────────────────────────────────────────────────────────────

class EscalarZScore(Transformador):
    """Escala columnas numéricas con z-score (media=0, std=1)."""

    def __init__(self, columnas):
        self.columnas = columnas
        self.scaler   = StandardScaler()

    def aplicar(self, df):
        cols = [c for c in self.columnas if c in df.columns]
        df[cols] = self.scaler.fit_transform(df[cols])
        logger.info(f"EscalarZScore aplicado a: {cols}")
        return df


class EscalarMinMax(Transformador):
    """Escala columnas numéricas al rango [0, 1]."""

    def __init__(self, columnas):
        self.columnas = columnas
        self.scaler   = MinMaxScaler()

    def aplicar(self, df):
        cols = [c for c in self.columnas if c in df.columns]
        df[cols] = self.scaler.fit_transform(df[cols])
        logger.info(f"EscalarMinMax aplicado a: {cols}")
        return df